# 01 — Person detection: fine-tune YOLO on VisDrone (aerial person)
**Goal (foundation):** quantify the aerial domain gap for detection and close
part of it by fine-tuning. Produces the before/after AP50 table used in the
report. ~40 min on a free T4.


### Setup (every notebook starts with this)
1. Runtime → Change runtime type → **T4 GPU** (free tier).
2. Zip your local `Project/` folder's `src/` and `scripts/` dirs as `src.zip`
   (`cd Project && zip -r src.zip src scripts`), then either upload it below
   or put it in Drive and adjust `SRC_ZIP`.


In [ ]:
# --- environment ---
!pip -q install ultralytics rtmlib onnxruntime-gpu
import torch, os
print('cuda:', torch.cuda.is_available())

# --- project code: upload src.zip (or mount Drive and set SRC_ZIP) ---
from pathlib import Path
SRC_ZIP = None  # e.g. '/content/drive/MyDrive/sar_project/src.zip'
if SRC_ZIP is None:
    from google.colab import files
    up = files.upload()  # choose src.zip
    SRC_ZIP = next(iter(up))
!mkdir -p /content/project && unzip -q -o "$SRC_ZIP" -d /content/project
import sys
sys.path.insert(0, '/content/project/src')
sys.path.insert(0, '/content/project')
print('project code ready')

# --- results go to Drive so they survive the session ---
from google.colab import drive
drive.mount('/content/drive')
OUT = Path('/content/drive/MyDrive/sar_project_results'); OUT.mkdir(parents=True, exist_ok=True)


In [ ]:
# VisDrone-DET from the ultralytics mirror (verified live)
%cd /content
!curl -sL -o vd_train.zip https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-train.zip
!curl -sL -o vd_val.zip   https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-val.zip
!mkdir -p data/raw && unzip -q vd_train.zip -d data/raw && unzip -q vd_val.zip -d data/raw && rm vd_*.zip


In [ ]:
# Convert to YOLO person-only format using the project converter
import config
from pathlib import Path
config.RAW_DIR = Path('/content/data/raw')
config.DATA_DIR = Path('/content/data')
config.VISDRONE_TRAIN = config.RAW_DIR / 'VisDrone2019-DET-train'
config.VISDRONE_VAL = config.RAW_DIR / 'VisDrone2019-DET-val'
config.VISDRONE_PERSON = config.DATA_DIR / 'visdrone_person'
from data import visdrone
visdrone.VISDRONE_PERSON = config.VISDRONE_PERSON
visdrone.DATA_DIR = config.DATA_DIR
visdrone.convert_split(config.VISDRONE_TRAIN, 'train', config.VISDRONE_PERSON)
visdrone.convert_split(config.VISDRONE_VAL, 'val', config.VISDRONE_PERSON)
yaml_path = visdrone.write_dataset_yaml(config.VISDRONE_PERSON)


In [ ]:
# Baseline: zero-shot COCO yolo11s on aerial imagery
import eval_detect
eval_detect.VISDRONE_PERSON = config.VISDRONE_PERSON
base = eval_detect.evaluate_on_visdrone('yolo11s.pt', imgsz=1280,
                                        tag='yolo11s zero-shot (COCO)', device=0)


In [ ]:
# Fine-tune (full train split). ~30 min on T4 at imgsz 1280.
from ultralytics import YOLO
model = YOLO('yolo11s.pt')
model.train(data=str(yaml_path), epochs=30, imgsz=1280, batch=8, device=0,
            project='/content/runs', name='yolo11s_visdrone_ft', exist_ok=True)
best = '/content/runs/yolo11s_visdrone_ft/weights/best.pt'


In [ ]:
# After: same evaluator, same split
ft = eval_detect.evaluate_on_visdrone(best, imgsz=1280,
                                      tag='yolo11s fine-tuned (30ep)', device=0)
import pandas as pd
df = pd.DataFrame([base, ft])
df.to_csv(OUT / 'detection_visdrone.csv', index=False)
!cp {best} {OUT}/yolo11s_visdrone_best.pt
df


**Report:** the AP50 jump zero-shot → fine-tuned *is* the detection domain-gap
result. Also try `--imgsz 640` vs `1280` as the resolution ablation.
